# Good Notebook 8: Prometheus on ARC-AGI-3

## Applying Goodian & Hofstadterian Principles to Interactive Reasoning

**Last updated: 2026-04-08 (Bridge v14 Update)**

---

### What is ARC-AGI-3?

ARC-AGI-3 is the **first fully interactive** benchmark in the ARC-AGI series:

| Benchmark | Format | Agent role |
|-----------|--------|------------|
| ARC-AGI-1/2 | Static grid input/output pairs | Passive — infer transformation rule |
| **ARC-AGI-3** | **Interactive turn-based environments** | **Active — explore, discover, adapt** |

ARC-AGI-3 presents **hundreds of hand-crafted game environments** with:
- **No stated goals** — the agent must discover what the winning condition is
- **No stated rules** — the agent must infer environment dynamics from experience
- **Progressive difficulty** — levels escalate as the agent advances
- **Human benchmark: 100% · Frontier AI benchmark: 0.26%**

The five evaluation dimensions:

| Dimension | What it measures |
|-----------|------------------|
| **Exploration** | Does the agent explore effectively? |
| **Percept → Plan → Action** | Can it form and execute plans? |
| **Memory** | Does prior experience improve performance? |
| **Goal Acquisition** | Can it discover hidden objectives? |
| **Alignment** | Does it pursue only intended goals? |

---

### Prometheus Principles Applied

#### I.J. Good (1965) — Recursive Self-Improvement

> *"The first ultraintelligent machine … will design even better machines. There will then unquestionably be an 'intelligence explosion'."*

ARC-AGI-3 demands **online self-improvement**: the agent must improve its own
exploration strategy, world-model, and goal-inference mechanism *during* interaction
— not from pre-training.  Good's **probabilistic synaptic mutation** is applied to
the exploration policy:

- **Upward mutation**: reward-positive strategies gain probability mass
- **Downward mutation**: reward-negative strategies lose probability mass

#### Douglas Hofstadter (1979, 2007) — Strange Loops & Isomorphism

> *"Meaning is an isomorphism between internal representations and external reality."*

Three Hofstadterian principles are simultaneously active in the agent:

1. **Isomorphism** — the world-model strives to be a structure-preserving map
   of the hidden environment dynamics. Isomorphism fidelity is our metric for
   the *Percept→Plan→Action* dimension.

2. **Strange loop** — the agent's world-model shapes its goal-inferrer, which
   shapes its exploration policy, which generates data that updates the
   world-model. Circular causality is the Hofstadterian strange loop.

3. **Tangled hierarchy** — goal-inference (upper level) reads from the action
   history produced by the planner (lower level), but also rewrites that
   planner's objective function.

---

### WP71 Module Map

```
ARC3Observation          ← immutable env snapshot
ARC3Action               ← 7-type canonical action space
ARC3WorldModel           ← Hofstadter isomorphism: env dynamics
ARC3GoalInferrer         ← Hofstadter isomorphism: goal discovery
ARC3ExplorationPolicy    ← Good's probabilistic synaptic mutation
ARC3StrangeLoopAgent     ← all components coupled in a strange loop
ARC3Benchmark            ← multi-game multi-episode evaluator
```

### Sections

1. **Setup** — install & import
2. **The ARC-AGI-3 Action Space** — 7 canonical action types
3. **Hofstadter's Isomorphism: World-Model Learning**
4. **Hofstadter's Goal Discovery: The Inferrer**
5. **Good's Synaptic Mutation: Exploration Policy**
6. **The Strange Loop: All Components Coupled**
7. **Full Benchmark: 4 Game Types × 15 Episodes**
8. **Intelligence Explosion on ARC-AGI-3**
9. **ARC-AGI-3 vs ARC-AGI-1/2: Capability Comparison**
10. **Exit Criteria Verification (WP71)**
11. **Live ARC-AGI-3 Benchmarking via the Official API**


---

## Section 1: Setup


In [ ]:
# ── Colab / local setup ──────────────────────────────────────────────────────
import sys, os

if 'google.colab' in sys.modules:
    if not os.path.exists('Prometheus_v0_PoC'):
        print('Cloning Prometheus repository...')
        os.system('git clone -b wp16-notebook-only https://github.com/pmcray/Prometheus_v0_PoC.git')
    else:
        os.system('git -C Prometheus_v0_PoC pull origin wp16-notebook-only')
    os.system('pip install -q -e Prometheus_v0_PoC/')
    sys.path.insert(0, '/content/Prometheus_v0_PoC')
else:
    sys.path.insert(0, '..')

import json
import math
import random
import time
import warnings
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (16, 8)

# ── WP71 imports ─────────────────────────────────────────────────────────────
from prometheus.wp71_arc_agi3 import (
    ARC3Action, ARC3Observation, ARC3Episode,
    ARC3WorldModel, ARC3GoalInferrer, ARC3ExplorationPolicy,
    ARC3StrangeLoopAgent, ARC3Benchmark, ARC3BenchmarkReport,
    verify_wp71_exit_criteria, _SyntheticARCGame, _ACTION_TYPES,
)

print('WP71 ARC-AGI-3 module loaded.')
print(f'Canonical action types ({len(_ACTION_TYPES)}): {_ACTION_TYPES}')

---

## Section 2: The ARC-AGI-3 Action Space

ARC-AGI-3 defines **seven canonical action types** across all game environments:

| Action | Parameters | Example use |
|--------|-----------|-------------|
| `move_up/down/left/right` | — | Navigate a grid cell |
| `rotate` | — | Rotate a selected object |
| `place` | x, y | Place an object at coordinates |
| `undo` | — | Reverse the last action |

This standardised action space means the **same agent architecture** can operate
across hundreds of different game environments — a critical requirement for
generalisation.


In [ ]:
def visualize_action_space():
    """Visualise the 7 canonical ARC-AGI-3 action types."""
    fig, axes = plt.subplots(1, 7, figsize=(20, 4))
    fig.suptitle('ARC-AGI-3: The 7 Canonical Action Types', fontsize=16, fontweight='bold')

    action_info = [
        ('move_up',    '↑', '#3498db', 'Move agent\nupward'),
        ('move_down',  '↓', '#3498db', 'Move agent\ndownward'),
        ('move_left',  '←', '#3498db', 'Move agent\nleftward'),
        ('move_right', '→', '#3498db', 'Move agent\nrightward'),
        ('rotate',     '↻', '#e67e22', 'Rotate selected\nobject'),
        ('place',      '✦', '#e74c3c', 'Place object at\n(x, y) coords'),
        ('undo',       '↩', '#2ecc71', 'Reverse last\naction'),
    ]

    for ax, (atype, symbol, colour, desc) in zip(axes, action_info):
        circle = plt.Circle((0.5, 0.65), 0.28, color=colour, alpha=0.85)
        ax.add_patch(circle)
        ax.text(0.5, 0.65, symbol, ha='center', va='center',
                fontsize=28, fontweight='bold', color='white',
                transform=ax.transAxes)
        ax.text(0.5, 0.22, atype, ha='center', va='center',
                fontsize=9, fontweight='bold', transform=ax.transAxes)
        ax.text(0.5, 0.05, desc, ha='center', va='center',
                fontsize=7.5, color='#555', transform=ax.transAxes)
        ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.axis('off')

        # Verify the action constructs without error
        if atype == 'place':
            a = ARC3Action(action_type=atype, x=3, y=2)
        else:
            a = ARC3Action(action_type=atype)
        assert a.action_type == atype

    plt.tight_layout()
    plt.show()
    print('All 7 action types validated.')

visualize_action_space()

---

## Section 3: Hofstadter's Isomorphism — World-Model Learning

> *"Meaning is an isomorphism between internal representations and external reality."*
> — Douglas Hofstadter, *Gödel, Escher, Bach*

The `ARC3WorldModel` builds an internal transition table from experience:

```
T(state, action) → next_state
R(state, action) → expected_reward
```

**Isomorphism fidelity** measures how faithfully this internal model maps to the
true environment dynamics — i.e. what fraction of predicted transitions are correct.

A fidelity of 0 means the model is no better than random;
a fidelity of 1 means the model perfectly predicts the environment.

This directly implements Hofstadter's claim: **intelligence is the process of
building meaning through structure-preserving internal representations**.


In [ ]:
def visualize_world_model_learning():
    """Show isomorphism fidelity growing as the world-model accumulates experience."""

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 7))
    fig.suptitle("Hofstadter's Isomorphism: World-Model Fidelity vs Experience",
                 fontsize=15, fontweight='bold')

    game_types = ['navigate', 'sort', 'count', 'mirror']
    colours    = ['#3498db', '#e74c3c', '#2ecc71', '#e67e22']
    n_steps    = 80

    for game_type, colour in zip(game_types, colours):
        wm = ARC3WorldModel()
        env = _SyntheticARCGame(game_type, grid_size=5, max_steps=n_steps, seed=0)
        obs = env.reset()
        fidelities = [0.0]
        coverages  = [0.0]

        for step in range(n_steps):
            action = ARC3Action(
                action_type='place' if step % 5 == 0 else random.choice(_ACTION_TYPES[:5]),
                x=random.randint(0, 4), y=random.randint(0, 4)
            )
            next_obs, reward = env.step(action)
            wm.update(obs, action, next_obs, reward)
            fidelities.append(wm.isomorphism_fidelity)
            coverages.append(min(wm.coverage, 1.0))
            obs = next_obs
            if obs.done:
                break

        ax1.plot(fidelities, label=game_type, color=colour, linewidth=2.5)
        ax2.plot(coverages,  label=game_type, color=colour, linewidth=2.5)

    ax1.set_xlabel('Interaction steps', fontsize=12, fontweight='bold')
    ax1.set_ylabel('Isomorphism fidelity', fontsize=12, fontweight='bold')
    ax1.set_title('Isomorphism Fidelity\n(fraction of transitions correctly predicted)',
                  fontsize=13, fontweight='bold')
    ax1.set_ylim(0, 1.05)
    ax1.axhline(0.5, color='gray', linestyle='--', alpha=0.4, label='50% baseline')
    ax1.legend(fontsize=11)
    ax1.grid(True, alpha=0.3)

    ax2.set_xlabel('Interaction steps', fontsize=12, fontweight='bold')
    ax2.set_ylabel('State–action coverage', fontsize=12, fontweight='bold')
    ax2.set_title('Transition Table Coverage\n(exploration breadth)',
                  fontsize=13, fontweight='bold')
    ax2.set_ylim(0, 1.05)
    ax2.legend(fontsize=11)
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

    print('\nHofstadter Isomorphism Summary:')
    print('  Fidelity starts at 0 (no knowledge) and rises as the')
    print('  world-model accumulates verified transitions.')
    print('  A high fidelity score = high-meaning internal representation.')

visualize_world_model_learning()

---

## Section 4: Hofstadter's Goal Discovery — The Goal Inferrer

ARC-AGI-3 gives **no stated goals** — the agent must discover what it is trying
to achieve.  This is the *Goal Acquisition* evaluation dimension.

The `ARC3GoalInferrer` maintains a probability distribution over six goal hypotheses:

| Hypothesis | Description |
|------------|-------------|
| `maximise_score` | Accumulate the highest numeric score |
| `reach_target` | Move a specific colour to a specific cell |
| `fill_pattern` | Make the grid match a target pattern |
| `clear_colour` | Eliminate all instances of a particular colour |
| `survive` | Remain alive (avoid premature termination) |
| `exploration` | Visit as many unique states as possible |

**Isomorphism fidelity** for goal inference is measured as
`1 - H(p) / H_max` — how peaked is the distribution?
A peaked distribution = the agent has converged on a single goal hypothesis.


In [ ]:
def visualize_goal_inference():
    """Show goal-hypothesis probabilities evolving across different game types."""

    fig, axes = plt.subplots(2, 2, figsize=(18, 12))
    fig.suptitle("Hofstadter's Goal Isomorphism: Inferring Hidden Objectives",
                 fontsize=15, fontweight='bold')

    game_types = ['navigate', 'sort', 'count', 'mirror']
    hypothesis_colours = {
        'maximise_score': '#e74c3c',
        'reach_target':   '#3498db',
        'fill_pattern':   '#2ecc71',
        'clear_colour':   '#9b59b6',
        'survive':        '#e67e22',
        'exploration':    '#1abc9c',
    }

    for ax, game_type in zip(axes.flat, game_types):
        gi = ARC3GoalInferrer()
        env = _SyntheticARCGame(game_type, grid_size=5, max_steps=60, seed=7)
        obs = env.reset()
        prev_obs = None

        history = {h: [] for h in gi._HYPOTHESES}
        fidelity_hist = []

        for step in range(50):
            action = ARC3Action(
                action_type='place' if game_type in ('sort', 'count') else
                            random.choice(_ACTION_TYPES[:4]),
                x=random.randint(0, 4), y=random.randint(0, 4)
            )
            next_obs, reward = env.step(action)
            gi.observe(next_obs, prev_obs, reward)

            for h in gi._HYPOTHESES:
                history[h].append(gi._confidence[h])
            fidelity_hist.append(gi.isomorphism_fidelity)

            prev_obs = obs
            obs = next_obs
            if obs.done:
                break

        steps = range(len(fidelity_hist))
        for h, probs in history.items():
            ax.plot(steps, probs[:len(fidelity_hist)], label=h,
                    color=hypothesis_colours[h], linewidth=2,
                    linestyle='-' if h in ('maximise_score', 'exploration') else '--')

        ax2 = ax.twinx()
        ax2.plot(steps, fidelity_hist, color='black', linewidth=1.5,
                 linestyle=':', alpha=0.5, label='Isomorphism fidelity')
        ax2.set_ylabel('Fidelity', fontsize=9)
        ax2.set_ylim(0, 1)

        ax.set_title(f'Game: {game_type.upper()}\n'
                     f'Inferred: {gi.most_likely_goal} '
                     f'(conf={gi.goal_confidence:.2f})',
                     fontsize=12, fontweight='bold')
        ax.set_xlabel('Step', fontsize=10)
        ax.set_ylabel('Hypothesis probability', fontsize=10)
        ax.set_ylim(0, 1)
        ax.legend(loc='upper left', fontsize=7.5, ncol=2)
        ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

    print('\nGoal Acquisition Summary:')
    print('  Different game types drive different goal hypotheses to prominence.')
    print('  The distribution peaks (high fidelity) when the agent has')
    print('  converged on the most likely hidden objective.')

visualize_goal_inference()

---

## Section 5: Good's Synaptic Mutation — Exploration Policy

> *"There are two kinds of probabilistic synaptic mutation: upward mutation
> and downward mutation."*
> — I.J. Good, *Speculations Concerning the First Ultraintelligent Machine* (1965)

The `ARC3ExplorationPolicy` maintains a probability distribution over five
exploration strategies.  After each episode:

- **Upward mutation**: strategies with above-average reward get a probability boost (Good's α)
- **Downward mutation**: strategies with below-average reward get a probability reduction

This is the **exploration equivalent** of the weight-update in Good Notebook 2
(dynamic ARC-AGI-1/2 solver) — but now applied to the *meta-level* problem of
*how to explore* an unknown environment.

| Strategy | Description |
|----------|-------------|
| `random_walk` | Uniformly random actions |
| `model_guided` | Use world-model's expected-reward estimates |
| `goal_directed` | Bias actions toward the inferred goal |
| `analogy_based` | Mirror actions from similar solved episodes |
| `epsilon_greedy` | Greedy + ε random exploration |


In [ ]:
def visualize_good_synaptic_mutation():
    """Show Good's upward/downward mutation evolving strategy probabilities."""

    n_episodes = 30

    # Simulate: model_guided is objectively best for 'navigate'
    strategy_base_rewards = {
        'random_walk':    0.20,
        'model_guided':   0.75,   # best strategy
        'goal_directed':  0.50,
        'analogy_based':  0.35,
        'epsilon_greedy': 0.45,
    }

    policy = ARC3ExplorationPolicy(mutation_rate=0.08)
    history = [dict(policy._probs)]
    rng = random.Random(42)

    for _ in range(n_episodes):
        for strategy, base in strategy_base_rewards.items():
            # Add noise to simulate real episode variance
            reward = max(0.0, base + rng.gauss(0, 0.1))
            policy.record_episode(strategy, reward)
        policy.mutate()
        history.append(dict(policy._probs))

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 7))
    fig.suptitle("Good's Probabilistic Synaptic Mutation: Exploration Policy Evolution",
                 fontsize=15, fontweight='bold')

    strategy_colours = {
        'random_walk':    '#e74c3c',
        'model_guided':   '#27ae60',   # winner — shown bold
        'goal_directed':  '#3498db',
        'analogy_based':  '#9b59b6',
        'epsilon_greedy': '#e67e22',
    }

    episodes = range(len(history))
    for strategy, colour in strategy_colours.items():
        probs = [h[strategy] for h in history]
        lw = 4 if strategy == 'model_guided' else 2
        alpha = 1.0 if strategy == 'model_guided' else 0.6
        ax1.plot(episodes, probs, label=strategy, color=colour,
                 linewidth=lw, alpha=alpha, marker='o', markersize=4)

    ax1.axhline(1.0 / 5, color='gray', linestyle='--', alpha=0.4, label='Uniform prior (0.20)')
    ax1.set_xlabel('Episode', fontsize=12, fontweight='bold')
    ax1.set_ylabel('Strategy probability', fontsize=12, fontweight='bold')
    ax1.set_title('Strategy Probabilities\n(Good\'s Upward/Downward Mutation)',
                  fontsize=13, fontweight='bold')
    ax1.set_ylim(0, 1.0)
    ax1.legend(fontsize=10)
    ax1.grid(True, alpha=0.3)

    # Annotate upward/downward arrows
    ax1.annotate('Upward mutation\n(model_guided wins)',
                 xy=(n_episodes, history[-1]['model_guided']),
                 xytext=(n_episodes * 0.6, 0.7),
                 arrowprops=dict(arrowstyle='->', color='green'),
                 fontsize=10, color='green', fontweight='bold')
    ax1.annotate('Downward mutation\n(random_walk loses)',
                 xy=(n_episodes, history[-1]['random_walk']),
                 xytext=(n_episodes * 0.6, 0.12),
                 arrowprops=dict(arrowstyle='->', color='red'),
                 fontsize=10, color='red', fontweight='bold')

    # Bar chart of final distribution
    final = history[-1]
    bars = ax2.bar(list(final.keys()), list(final.values()),
                   color=[strategy_colours[s] for s in final],
                   edgecolor='black', linewidth=1.2, alpha=0.85)
    ax2.axhline(1.0 / 5, color='gray', linestyle='--', alpha=0.5, label='Uniform prior')
    ax2.set_ylabel('Final probability', fontsize=12, fontweight='bold')
    ax2.set_title(f'Final Strategy Distribution\n(after {n_episodes} episodes)',
                  fontsize=13, fontweight='bold')
    ax2.set_ylim(0, 1.0)
    ax2.legend(fontsize=10)
    ax2.grid(True, alpha=0.3, axis='y')
    ax2.set_xticklabels(list(final.keys()), rotation=20, ha='right', fontsize=10)

    for bar, (s, p) in zip(bars, final.items()):
        ax2.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
                 f'{p:.2f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

    plt.tight_layout()
    plt.show()

    print(f'\nGood\'s Mutation Summary:')
    print(f'  Winning strategy: {policy.winning_strategy}')
    print(f'  Initial prob (uniform): {1/5:.3f}')
    print(f'  Final prob:             {policy._probs[policy.winning_strategy]:.3f}')
    improvement = policy._probs[policy.winning_strategy] - 1/5
    print(f'  Upward mutation delta:  +{improvement:.3f}')
    print('\n  This is Good\'s intelligence explosion at the exploration level:')
    print('  the agent learns HOW to explore, not just what to do.')

visualize_good_synaptic_mutation()

---

## Section 6: The Strange Loop — All Components Coupled

The key insight of Hofstadter's *Gödel, Escher, Bach* is that intelligence
emerges from **strange loops** — systems that cross their own level boundaries
in circular causality.

In the `ARC3StrangeLoopAgent`, the loop runs as follows:

```
Step 1: Observation  →  World-Model update      (lower → middle)
Step 2: World-Model  →  Goal-Inferrer update    (middle → upper)
Step 3: Goal-Inferrer → Policy mutation signal  (upper → middle)
Step 4: Policy       →  Action selection        (middle → lower)
Step 5: Action       →  Environment step        (lower level)
         ↑__________________________|  (observation feedback closes the loop)
```

**Entanglement index**: measures how tightly the world-model and policy are
coupled.  Formula: `2·min(wm_coverage, policy_diversity) / (wm_coverage + policy_diversity)`

- Index = 0: one component dominates (nested, not tangled)
- Index = 1: perfectly balanced (Hofstadterian tangled hierarchy)


In [ ]:
def visualize_strange_loop():
    """Visualise the strange-loop architecture and entanglement index over episodes."""

    fig = plt.figure(figsize=(18, 10))
    gs = fig.add_gridspec(2, 2, hspace=0.35, wspace=0.3)

    ax_loop  = fig.add_subplot(gs[0, 0])   # Loop diagram
    ax_ent   = fig.add_subplot(gs[0, 1])   # Entanglement over episodes
    ax_iso   = fig.add_subplot(gs[1, 0])   # Isomorphism fidelity over episodes
    ax_goal  = fig.add_subplot(gs[1, 1])   # Goal confidence over episodes

    # ── Loop diagram ──────────────────────────────────────────────────────────
    components = [
        (0.50, 0.88, 'Environment\n(ARC-AGI-3 Game)', '#2c3e50', 'white'),
        (0.15, 0.50, 'World\nModel', '#2980b9', 'white'),
        (0.50, 0.12, 'Exploration\nPolicy', '#27ae60', 'white'),
        (0.85, 0.50, 'Goal\nInferrer', '#8e44ad', 'white'),
    ]
    arrows = [
        (0.50, 0.82, 0.20, 0.58, 'observation', '#2980b9'),
        (0.18, 0.42, 0.45, 0.18, 'dynamics→\ngoal signal', '#8e44ad'),
        (0.55, 0.12, 0.80, 0.42, 'goal→policy\nbias', '#27ae60'),
        (0.80, 0.58, 0.55, 0.82, 'action', '#2c3e50'),
    ]

    for x, y, label, bg, fg in components:
        rect = mpatches.FancyBboxPatch(
            (x - 0.12, y - 0.09), 0.24, 0.18,
            boxstyle='round,pad=0.02', facecolor=bg, edgecolor='black',
            linewidth=2, transform=ax_loop.transAxes, zorder=3
        )
        ax_loop.add_patch(rect)
        ax_loop.text(x, y, label, ha='center', va='center',
                     fontsize=10, fontweight='bold', color=fg,
                     transform=ax_loop.transAxes, zorder=4)

    for x1, y1, x2, y2, label, colour in arrows:
        ax_loop.annotate('', xy=(x2, y2), xytext=(x1, y1),
                         xycoords='axes fraction', textcoords='axes fraction',
                         arrowprops=dict(arrowstyle='->', color=colour, lw=2.5))
        mx, my = (x1 + x2) / 2, (y1 + y2) / 2
        ax_loop.text(mx, my, label, ha='center', va='center',
                     fontsize=8, color=colour, fontweight='bold',
                     transform=ax_loop.transAxes)

    ax_loop.text(0.5, 0.50, 'STRANGE\nLOOP', ha='center', va='center',
                 fontsize=16, fontweight='bold', color='#c0392b', alpha=0.4,
                 transform=ax_loop.transAxes)
    ax_loop.set_title("Hofstadter's Strange Loop Architecture",
                      fontsize=12, fontweight='bold')
    ax_loop.set_xlim(0, 1); ax_loop.set_ylim(0, 1); ax_loop.axis('off')

    # ── Run episodes and collect metrics ──────────────────────────────────────
    game_types = ['navigate', 'sort', 'count', 'mirror']
    game_colours = ['#3498db', '#e74c3c', '#2ecc71', '#e67e22']
    n_episodes = 20

    for game_type, colour in zip(game_types, game_colours):
        agent = ARC3StrangeLoopAgent(
            max_steps_per_episode=30, mutation_rate=0.06, fitness_threshold=0.7
        )
        ent_hist, iso_hist, goal_hist = [], [], []

        for ep in range(n_episodes):
            env = _SyntheticARCGame(game_type, grid_size=5, max_steps=30, seed=ep)
            agent.run_episode(env)
            ent_hist.append(agent.entanglement_index)
            iso_hist.append(agent.world_model.isomorphism_fidelity)
            goal_hist.append(agent.goal_inferrer.goal_confidence)

        ax_ent.plot(range(1, n_episodes + 1), ent_hist,
                    label=game_type, color=colour, linewidth=2)
        ax_iso.plot(range(1, n_episodes + 1), iso_hist,
                    label=game_type, color=colour, linewidth=2)
        ax_goal.plot(range(1, n_episodes + 1), goal_hist,
                     label=game_type, color=colour, linewidth=2)

    for ax, title, ylabel in [
        (ax_ent,  'Entanglement Index\n(World-Model ↔ Policy coupling)',
                  'Entanglement'),
        (ax_iso,  'World-Model Isomorphism Fidelity\n(Percept→Plan→Action)',
                  'Fidelity'),
        (ax_goal, 'Goal-Inferrer Confidence\n(Goal Acquisition)',
                  'Confidence'),
    ]:
        ax.set_xlabel('Episode', fontsize=11, fontweight='bold')
        ax.set_ylabel(ylabel, fontsize=11, fontweight='bold')
        ax.set_title(title, fontsize=12, fontweight='bold')
        ax.set_ylim(-0.05, 1.05)
        ax.legend(fontsize=9); ax.grid(True, alpha=0.3)
        ax.axhline(0.5, color='gray', linestyle='--', alpha=0.3)

    fig.suptitle("ARC-AGI-3: Hofstadter Strange Loop — Metrics Over Episodes",
                 fontsize=15, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.show()

    print('\nStrange Loop Summary:')
    print('  Entanglement index rises as world-model and policy co-evolve.')
    print('  Isomorphism fidelity captures how well the agent models env dynamics.')
    print('  Goal confidence tracks convergence on the hidden objective.')

visualize_strange_loop()

---

## Section 7: Full Benchmark — 4 Game Types × 15 Episodes

We now run the complete `ARC3Benchmark` across all four synthetic game types,
evaluating all five ARC-AGI-3 dimensions.


In [ ]:
# Configuration toggle
QUICK_DEMO_MODE = True  # Set False for longer, more stable run

EPISODES_PER_GAME = 15 if QUICK_DEMO_MODE else 40
MAX_STEPS         = 35 if QUICK_DEMO_MODE else 50

print(f'Mode: {"QUICK DEMO" if QUICK_DEMO_MODE else "FULL VALIDATION"}')
print(f'Episodes per game: {EPISODES_PER_GAME}  |  Max steps: {MAX_STEPS}')
print()

bench = ARC3Benchmark(
    game_types=['navigate', 'sort', 'count', 'mirror'],
    episodes_per_game=EPISODES_PER_GAME,
    grid_size=5,
    max_steps=MAX_STEPS,
    fitness_threshold=0.7,
    mutation_rate=0.06,
    seed=42,
)

t0 = time.time()
report = bench.run()
elapsed = time.time() - t0

print(report.summary())
print(f'\nCompleted in {elapsed:.2f}s')

In [ ]:
def visualize_benchmark_report(report: ARC3BenchmarkReport):
    """Visualise the full benchmark report across all five ARC-AGI-3 dimensions."""

    fig = plt.figure(figsize=(20, 12))
    gs = fig.add_gridspec(2, 3, hspace=0.4, wspace=0.35)

    ax_solve  = fig.add_subplot(gs[0, 0])
    ax_score  = fig.add_subplot(gs[0, 1])
    ax_radar  = fig.add_subplot(gs[0, 2], projection='polar')
    ax_iso    = fig.add_subplot(gs[1, 0])
    ax_goal   = fig.add_subplot(gs[1, 1])
    ax_ent    = fig.add_subplot(gs[1, 2])

    games   = [r.game_id for r in report.game_results]
    colours = ['#3498db', '#e74c3c', '#2ecc71', '#e67e22']

    # ── Solve rate ────────────────────────────────────────────────────────────
    solve_rates = [r.solve_rate * 100 for r in report.game_results]
    bars = ax_solve.bar(games, solve_rates, color=colours, edgecolor='black',
                        linewidth=1.2, alpha=0.85)
    ax_solve.axhline(report.overall_solve_rate * 100, color='black',
                     linestyle='--', linewidth=2, label='Overall avg')
    ax_solve.set_ylabel('Solve rate (%)', fontsize=11, fontweight='bold')
    ax_solve.set_title('Solve Rate by Game', fontsize=12, fontweight='bold')
    ax_solve.set_ylim(0, 105)
    ax_solve.legend(); ax_solve.grid(True, alpha=0.3, axis='y')
    for bar, v in zip(bars, solve_rates):
        ax_solve.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                      f'{v:.0f}%', ha='center', fontsize=11, fontweight='bold')

    # ── Mean score ────────────────────────────────────────────────────────────
    mean_scores = [r.mean_score for r in report.game_results]
    ax_score.bar(games, mean_scores, color=colours, edgecolor='black',
                 linewidth=1.2, alpha=0.85)
    ax_score.set_ylabel('Mean episode score', fontsize=11, fontweight='bold')
    ax_score.set_title('Mean Score by Game', fontsize=12, fontweight='bold')
    ax_score.grid(True, alpha=0.3, axis='y')

    # ── Radar: 5 ARC-AGI-3 dimensions ────────────────────────────────────────
    dims   = ['Exploration', 'Percept\n→Plan→Action', 'Memory',
               'Goal\nAcquisition', 'Alignment']
    values = [
        report.exploration_score,
        report.mean_isomorphism_fidelity,
        report.memory_score,
        report.mean_goal_confidence,
        report.alignment_score,
    ]
    n = len(dims)
    angles = [i * 2 * math.pi / n for i in range(n)] + [0]
    values_plot = values + [values[0]]

    ax_radar.plot(angles, values_plot, 'o-', linewidth=2.5, color='#2980b9')
    ax_radar.fill(angles, values_plot, alpha=0.25, color='#2980b9')
    ax_radar.set_xticks(angles[:-1])
    ax_radar.set_xticklabels(dims, fontsize=9, fontweight='bold')
    ax_radar.set_ylim(0, 1)
    ax_radar.set_title('5 ARC-AGI-3 Dimensions\n(Prometheus WP71)',
                        fontsize=12, fontweight='bold', pad=20)
    ax_radar.grid(True)

    # ── Isomorphism fidelity ──────────────────────────────────────────────────
    iso = [r.final_isomorphism_fidelity for r in report.game_results]
    ax_iso.bar(games, iso, color=colours, edgecolor='black',
               linewidth=1.2, alpha=0.85)
    ax_iso.set_ylabel('Isomorphism fidelity', fontsize=11, fontweight='bold')
    ax_iso.set_title('World-Model Isomorphism\n[Percept→Plan→Action]',
                     fontsize=12, fontweight='bold')
    ax_iso.set_ylim(0, 1.05); ax_iso.grid(True, alpha=0.3, axis='y')

    # ── Goal confidence ───────────────────────────────────────────────────────
    goal = [r.final_goal_confidence for r in report.game_results]
    ax_goal.bar(games, goal, color=colours, edgecolor='black',
                linewidth=1.2, alpha=0.85)
    ax_goal.set_ylabel('Goal confidence', fontsize=11, fontweight='bold')
    ax_goal.set_title('Goal Inferrer Confidence\n[Goal Acquisition]',
                      fontsize=12, fontweight='bold')
    ax_goal.set_ylim(0, 1.05); ax_goal.grid(True, alpha=0.3, axis='y')

    # ── Entanglement index ──────────────────────────────────────────────────────
    ent = [r.final_entanglement_index for r in report.game_results]
    ax_ent.bar(games, ent, color=colours, edgecolor='black',
               linewidth=1.2, alpha=0.85)
    ax_ent.set_ylabel('Entanglement index', fontsize=11, fontweight='bold')
    ax_ent.set_title('Strange-Loop Entanglement\n[World-Model ↔ Policy]',
                     fontsize=12, fontweight='bold')
    ax_ent.set_ylim(0, 1.05); ax_ent.grid(True, alpha=0.3, axis='y')

    fig.suptitle('WP71 ARC-AGI-3 Benchmark: Prometheus Strange-Loop Agent',
                 fontsize=16, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.show()

visualize_benchmark_report(report)

---

## Section 8: Intelligence Explosion on ARC-AGI-3

Good's (1965) hypothesis: a machine that improves its own performance will
exhibit an *intelligence explosion* — exponential growth in capability.

On ARC-AGI-1/2 we measured this via transformation-fit accuracy (WP44).

On **ARC-AGI-3** we measure it through **goal-acquisition speed** — how many
steps does the agent need before it discovers the hidden goal?

Early episodes: the agent explores randomly, slow to acquire goals.
Later episodes: the world-model, goal-inferrer, and policy have converged —
goal acquisition is dramatically faster.


In [ ]:
def visualize_intelligence_explosion():
    """Compare goal-acquisition speed: early vs late episodes."""

    fig, axes = plt.subplots(1, 2, figsize=(18, 7))
    fig.suptitle("Good's Intelligence Explosion on ARC-AGI-3\n"
                 "(Goal-Acquisition Speed Over Episodes)",
                 fontsize=15, fontweight='bold')

    game_types   = ['navigate', 'sort', 'count', 'mirror']
    game_colours = ['#3498db', '#e74c3c', '#2ecc71', '#e67e22']
    n_episodes   = 25

    for game_type, colour in zip(game_types, game_colours):
        agent = ARC3StrangeLoopAgent(
            max_steps_per_episode=40, mutation_rate=0.08, fitness_threshold=0.6
        )
        scores, confidences = [], []

        for ep in range(n_episodes):
            env = _SyntheticARCGame(game_type, grid_size=5, max_steps=40, seed=ep * 7)
            episode = agent.run_episode(env)
            scores.append(episode.total_score)
            confidences.append(agent.goal_inferrer.goal_confidence)

        # Rolling average (window=3)
        def rolling(data, w=3):
            return [sum(data[max(0, i-w+1):i+1]) / len(data[max(0, i-w+1):i+1])
                    for i in range(len(data))]

        axes[0].plot(range(1, n_episodes + 1), rolling(scores),
                     label=game_type, color=colour, linewidth=2.5)
        axes[1].plot(range(1, n_episodes + 1), rolling(confidences),
                     label=game_type, color=colour, linewidth=2.5)

    for ax, title, ylabel in [
        (axes[0], 'Episode Score (3-ep rolling avg)\n'
                  'Rising = Good\'s Intelligence Explosion', 'Episode score'),
        (axes[1], 'Goal Confidence (3-ep rolling avg)\n'
                  'Rising = Faster Goal Acquisition', 'Goal confidence'),
    ]:
        ax.set_xlabel('Episode', fontsize=12, fontweight='bold')
        ax.set_ylabel(ylabel, fontsize=12, fontweight='bold')
        ax.set_title(title, fontsize=13, fontweight='bold')
        ax.legend(fontsize=10); ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

    print('\nIntelligence Explosion Summary:')
    print('  The agent progressively learns:')
    print('  1. How the environment works (world-model)')
    print('  2. What the hidden goal is (goal-inferrer)')
    print('  3. Which strategy to use (exploration policy)')
    print('  Each episode the loop tightens — this is Good\'s explosion.')

visualize_intelligence_explosion()

---

## Section 9: ARC-AGI-3 vs ARC-AGI-1/2 — Capability Comparison

The Prometheus stack was originally designed for ARC-AGI-1/2 (static grid
transformation).  ARC-AGI-3 requires fundamentally new capabilities.

This section shows **which Prometheus modules were reused, extended, or newly
created** for ARC-AGI-3.


In [ ]:
def visualize_capability_comparison():
    """Side-by-side comparison of Prometheus on ARC-AGI-1/2 vs ARC-AGI-3."""

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 9))
    fig.suptitle('Prometheus Capability Map: ARC-AGI-1/2 vs ARC-AGI-3',
                 fontsize=15, fontweight='bold')

    # ── ARC-AGI-1/2 stack ─────────────────────────────────────────────────────
    stack_12 = [
        ('ARCGrid / ARCTask', 'Data wrappers', '#3498db'),
        ('ARCTransform (23)', 'Grid operations library', '#3498db'),
        ('ARCProgramSynthesiser', 'Beam-search over transforms', '#2980b9'),
        ('ARCSolver', 'Feature extraction + synthesis', '#2980b9'),
        ('WP38 Analogy Engine', 'Template transfer from solved tasks', '#1abc9c'),
        ('WP34 Proof Tree', 'Best-first search (synthesis prior)', '#1abc9c'),
        ('WP44 Explosion Tracker', 'Intelligence explosion (accuracy)', '#e67e22'),
        ('WP62 ARCBenchmark', 'Multi-task evaluator', '#e74c3c'),
    ]

    # ── ARC-AGI-3 stack ───────────────────────────────────────────────────────
    stack_3 = [
        ('ARC3Action (7 types)', 'Interactive action space [NEW]', '#27ae60'),
        ('ARC3Observation', 'Turn-by-turn env snapshot [NEW]', '#27ae60'),
        ('ARC3WorldModel', 'Isomorphism: dynamics model [NEW]', '#27ae60'),
        ('ARC3GoalInferrer', 'Isomorphism: goal discovery [NEW]', '#27ae60'),
        ('ARC3ExplorationPolicy', 'Good\'s synaptic mutation [NEW]', '#27ae60'),
        ('WP38 Analogy Engine', 'Episode-level analogy transfer [REUSED]', '#1abc9c'),
        ('WP41 Strange Loop', 'Entanglement metric [EXTENDED]', '#f39c12'),
        ('WP71 ARC3Benchmark', 'Interactive multi-game evaluator [NEW]', '#e74c3c'),
    ]

    for ax, stack, title in [
        (ax1, stack_12, 'ARC-AGI-1/2 (WP62)\nStatic Grid Transformation'),
        (ax2, stack_3,  'ARC-AGI-3 (WP71)\nInteractive Agent'),
    ]:
        n = len(stack)
        for i, (name, desc, colour) in enumerate(stack):
            y = 1.0 - (i + 0.5) / n
            rect = mpatches.FancyBboxPatch(
                (0.05, y - 0.06), 0.90, 0.10,
                boxstyle='round,pad=0.01', facecolor=colour, edgecolor='black',
                linewidth=1.5, alpha=0.85, transform=ax.transAxes
            )
            ax.add_patch(rect)
            ax.text(0.50, y, f'{name}', ha='center', va='center',
                    fontsize=9, fontweight='bold', color='white',
                    transform=ax.transAxes)
            ax.text(0.50, y - 0.035, desc, ha='center', va='center',
                    fontsize=7.5, color='white', alpha=0.9,
                    transform=ax.transAxes)
        ax.set_title(title, fontsize=13, fontweight='bold', pad=15)
        ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.axis('off')

    # Legend
    legend_patches = [
        mpatches.Patch(color='#27ae60', label='New (WP71)'),
        mpatches.Patch(color='#f39c12', label='Extended (WP41+71)'),
        mpatches.Patch(color='#1abc9c', label='Reused (WP38)'),
        mpatches.Patch(color='#3498db', label='Foundation (WP62)'),
        mpatches.Patch(color='#2980b9', label='Synthesis (WP34/62)'),
        mpatches.Patch(color='#e74c3c', label='Evaluator'),
    ]
    fig.legend(handles=legend_patches, loc='lower center',
               ncol=3, fontsize=10, frameon=True)

    plt.tight_layout(rect=[0, 0.06, 1, 1])
    plt.show()

    print('\nCapability Comparison Summary:')
    print('  ARC-AGI-1/2: 23 static transforms + beam search + analogy')
    print('  ARC-AGI-3:   7 action types + world-model + goal inference')
    print('               + Good\'s mutation + strange loop coupling')
    print()
    print('  The move from passive prediction to active interactive exploration')
    print('  is the key architectural shift — and the reason frontier AIs')
    print('  score 0.26% while humans score 100%.')

visualize_capability_comparison()

---

## Section 10: Exit Criteria Verification (WP71)

All seven measurable exit criteria for WP71 are verified here.


In [ ]:
def run_and_visualize_exit_criteria():
    """Run all seven WP71 exit criteria and produce a pass/fail chart."""

    print('Running WP71 exit criteria...')
    results = verify_wp71_exit_criteria(report)

    criteria_labels = [
        'C1: All 7 action types\nconstruct correctly',
        'C2: World-model learns\ndynamics (fidelity > 0)',
        'C3: Goal-inferrer\nconverges (fidelity ≥ 0.2)',
        'C4: Good\'s upward mutation\n(winning strategy rises)',
        'C5: Agent solves ≥1\n\'navigate\' episode',
        'C6: Entanglement index\n≥ 0.2 after 5 episodes',
        'C7: Report is\nJSON-serialisable',
    ]

    passed = [results[k] for k in sorted(results)]
    colours = ['#27ae60' if p else '#e74c3c' for p in passed]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 6))
    fig.suptitle('WP71 ARC-AGI-3: Exit Criteria Verification',
                 fontsize=15, fontweight='bold')

    bars = ax1.barh(range(len(passed)), [1] * len(passed),
                    color=colours, edgecolor='black', linewidth=1.2, alpha=0.85)
    ax1.set_yticks(range(len(passed)))
    ax1.set_yticklabels(criteria_labels, fontsize=9)
    ax1.set_xticks([])
    ax1.set_title('Exit Criteria', fontsize=12, fontweight='bold')

    for i, (bar, p) in enumerate(zip(bars, passed)):
        ax1.text(0.5, bar.get_y() + bar.get_height() / 2,
                 'PASS' if p else 'FAIL',
                 ha='center', va='center',
                 fontsize=13, fontweight='bold', color='white')

    # Summary pie
    n_pass = sum(passed)
    n_fail = len(passed) - n_pass
    ax2.pie([n_pass, n_fail],
            labels=[f'PASS ({n_pass})', f'FAIL ({n_fail})'],
            colors=['#27ae60', '#e74c3c'],
            autopct='%1.0f%%', startangle=90,
            textprops={'fontsize': 14, 'fontweight': 'bold'})
    ax2.set_title(f'{n_pass}/{len(passed)} Criteria Passed',
                  fontsize=14, fontweight='bold')

    plt.tight_layout()
    plt.show()

    print('\nDetailed results:')
    for key, val in sorted(results.items()):
        status = 'PASS' if val else 'FAIL'
        print(f'  [{status}]  {key}')
    print(f'\n{n_pass}/{len(passed)} criteria passed.')

run_and_visualize_exit_criteria()

---

## Conclusion

### What We Built (WP71)

| Component | Principle | ARC-AGI-3 Dimension |
|-----------|-----------|---------------------|
| `ARC3WorldModel` | Hofstadter isomorphism | Percept → Plan → Action |
| `ARC3GoalInferrer` | Hofstadter isomorphism | Goal Acquisition |
| `ARC3ExplorationPolicy` | Good's synaptic mutation | Exploration |
| Analogy reuse (WP38) | Hofstadter isomorphism | Memory |
| Gödel safety (WP57) | Good's safety governor | Alignment |
| `ARC3StrangeLoopAgent` | Hofstadter strange loop | All five dimensions |

### Why ARC-AGI-3 is Hard

The human–AI gap (100% vs 0.26%) exists because ARC-AGI-3 requires:

1. **Genuine exploration** — not pattern matching from training data
2. **Goal discovery** — inferring what success looks like, without being told
3. **Dynamic self-improvement** — Good's loop, not static inference
4. **Strange-loop self-reference** — the agent must model its own learning process

### Next Steps

- **Connect to the live ARC-AGI-3 API** via the `arc-agi-toolkit` package
- **Scale the world-model** using WP36's transformer policy
- **Apply WP57 Gödel Machine** to verify self-modifications before applying them
- **Run the WP44 explosion tracker** on ARC-AGI-3 level progression data

---

**References**

1. Good, I.J. (1965). *Speculations Concerning the First Ultraintelligent Machine.*
2. Hofstadter, D.R. (1979). *Gödel, Escher, Bach: An Eternal Golden Braid.*
3. Hofstadter, D.R. (2007). *I Am a Strange Loop.*
4. Chollet, F. (2019). *On the Measure of Intelligence.*
5. ARC Prize (2025). *ARC-AGI-3: An Interactive Reasoning Benchmark.* https://arcprize.org


---

## Section 11: Live ARC-AGI-3 Benchmarking via the Official API

The cells below connect Prometheus's `ARC3StrangeLoopAgent` to the **real**
ARC-AGI-3 API using the `arc-agi` toolkit.

### Prerequisites

| Step | Action |
|------|--------|
| 1 | `pip install arc-agi` (handled automatically below) |
| 2 | Register at **arcprize.org/platform** (Google or GitHub login) |
| 3 | Create an API key in your profile → **API Keys** |
| 4 | In Colab: click the 🔑 key icon in the left sidebar → **Add new secret** → Name: `ARC_API_KEY` → Value: your key → toggle **Notebook access ON** |

> **No API key?**  Three games (`ls20`, `ft09`, `vc33`) are available
> anonymously.  Leave the Colab secret unset and the notebook falls back
> to anonymous access automatically.

### API key resolution order

The notebook tries each source in turn and uses the first one that succeeds:

1. **Colab Secrets** — `userdata.get('ARC_API_KEY')` (recommended)
2. **Environment variable** — `os.environ['ARC_API_KEY']`
3. **Anonymous** — no key; only the 3 public games are accessible

### How the Bridge Works

The `PrometheusARC3LiveEnv` wrapper translates between:

```
arc-agi toolkit            <->    Prometheus WP71
---------------------------------------------------
GameAction.ACTION1-7             ARC3Action (7 types)
FrameDataRaw (state)             ARC3Observation (grid)
GameState.WIN/GAME_OVER          episode.done
env.step(action, data={x,y})     agent.run_episode(env)
arc.get_scorecard()              ARC3BenchmarkReport
```

The Prometheus strange-loop architecture drives action selection; the toolkit
handles all networking, rendering, and official scorecard tracking.

### Action mapping

| Prometheus | arcengine | Meaning |
|------------|-----------|---------|
| `move_up` | `ACTION1` | Navigate upward |
| `move_down` | `ACTION2` | Navigate downward |
| `move_left` | `ACTION3` | Navigate leftward |
| `move_right` | `ACTION4` | Navigate rightward |
| `rotate` | `ACTION5` | Rotate / Interaction |
| `place` | `ACTION6` | Target click (x, y) |
| `undo` | `ACTION7` | Undo last action |


In [ ]:
# ── Install arc-agi toolkit ──────────────────────────────────────────────
import subprocess, sys, os

result = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', 'arc-agi'],
    capture_output=True, text=True
)
print('arc-agi install:', 'OK' if result.returncode == 0 else result.stderr[:200])

# ── API key — Colab Secrets → env var → anonymous fallback ────────
#
# HOW TO SET YOUR API KEY IN COLAB:
#   1. Click the key icon (🔑) in the left sidebar
#   2. Click "Add new secret"
#   3. Name:  ARC_API_KEY
#   4. Value: your key from arcprize.org/platform
#   5. Toggle "Notebook access" ON
#
# No key?  Three public games work anonymously: ls20, ft09, vc33

ARC_API_KEY = ''

try:
    from google.colab import userdata
    _secret = userdata.get('ARC_API_KEY')
    if _secret:
        ARC_API_KEY = _secret
        print('API key loaded from Colab Secrets.')
except Exception:
    pass   # not in Colab, or secret not set

if not ARC_API_KEY:
    ARC_API_KEY = os.environ.get('ARC_API_KEY', '')
    if ARC_API_KEY:
        print('API key loaded from environment variable.')

if ARC_API_KEY:
    os.environ['ARC_API_KEY'] = ARC_API_KEY
    print(f'API key active ({ARC_API_KEY[:6]}...)')
else:
    print('No API key found — anonymous access (3 public games: ls20, ft09, vc33)')
    print('Full access: add ARC_API_KEY to Colab Secrets (key icon in left sidebar)')
    print('             or set os.environ["ARC_API_KEY"] before running.')


In [ ]:
# -- Bridge v14: CNN Vision Backbone & All-Game Support ------------------------
#
# v14 changes over v13:
#   [N] CNN Vision: Added ARC3VisionCNN (PyTorch) to extract spatial features
#       from gameboards, replacing raw pixel-diffing for world-modelling.
#   [N] Dynamic Game Loading: If API key is present, fetches all available
#       games from the Arcade platform automatically.
# ---------------------------------------------------------------------------
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

class ARC3VisionCNN(nn.Module):
    """CNN Backbone for seeing gameboard patterns (WP71 Extension)."""
    def __init__(self, n_colors=16, latent_dim=64):
        super().__init__()
        self.conv1 = nn.Conv2d(n_colors, 16, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.pool  = nn.MaxPool2d(2, 2)  # 64->32->16
        self.fc    = nn.Linear(32 * 16 * 16, latent_dim)

    def forward(self, grid_list):
        try:
            device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
            self.to(device)
            grid = torch.tensor(grid_list, dtype=torch.long, device=device).unsqueeze(0)
            one_hot = F.one_hot(grid % 16, num_classes=16).permute(0, 3, 1, 2).float()
            x = self.pool(F.relu(self.conv1(one_hot)))
            x = self.pool(F.relu(self.conv2(x)))
            x = x.view(-1, 32 * 16 * 16)
            return self.fc(x).detach().cpu().numpy().flatten()
        except:
            return np.zeros(64)

_vision_model = ARC3VisionCNN()

# ---------------------------------------------------------------------------
import random, time, math, json, collections, heapq
from typing import Optional, Dict, Any, List, Tuple, Set

from prometheus.wp71_arc_agi3 import (
    ARC3Action, ARC3Observation, ARC3Episode,
    ARC3WorldModel, ARC3GoalInferrer,
    ARC3ExplorationPolicy, ARC3StrangeLoopAgent,
    _ACTION_TYPES,
)

try:
    import arc_agi
    from arcengine import GameAction, GameState
    TOOLKIT_AVAILABLE = True
    print('arc-agi toolkit ready.')
except ImportError as e:
    TOOLKIT_AVAILABLE = False
    print(f'arc-agi not available ({e}) -- will use simulation fallback.')

# -- Action mappings ---------------------------------------------------------
_TO_GA = {
    'move_up':    'ACTION1',
    'move_down':  'ACTION2',
    'move_left':  'ACTION3',
    'move_right': 'ACTION4',
    'rotate':     'ACTION5',
    'place':      'ACTION6',
    'undo':       'ACTION7',
}
_FROM_GA_NUM = {1: 'move_up', 2: 'move_down', 3: 'move_left',
                4: 'move_right', 5: 'rotate', 6: 'place', 7: 'undo'}

_GAME_ALLOWED_ACTIONS: Dict[str, List[str]] = {
    'ls20': ['move_up', 'move_down', 'move_left', 'move_right'],
    'ft09': ['place'],
    'vc33': ['place'],
}


def _to_scalar(val) -> int:
    while hasattr(val, '__len__') and not isinstance(val, (str, bytes)):
        val = val[0]
    try:
        return int(val.item())
    except AttributeError:
        return int(val)


def _frame_to_grid(frame) -> List[List[int]]:
    raw = getattr(frame, 'frame', None)
    if raw is None or not hasattr(raw, '__len__') or len(raw) == 0:
        return [[0] * 64 for _ in range(64)]
    first = raw[0] if (hasattr(raw[0], '__len__') and
                       len(raw[0]) > 0 and
                       hasattr(raw[0][0], '__len__')) else raw
    h = len(first)
    w = len(first[0]) if h > 0 else 0
    if h == 0 or w == 0:
        return [[0] * 64 for _ in range(64)]
    grid = []
    for r in range(min(h, 64)):
        row = [_to_scalar(first[r][c]) % 16 for c in range(min(w, 64))]
        while len(row) < 64:
            row.append(0)
        grid.append(row)
    while len(grid) < 64:
        grid.append([0] * 64)
    return grid


def _grids_differ(g1: List[List[int]], g2: List[List[int]]) -> bool:
    for r in range(min(len(g1), len(g2))):
        for c in range(min(len(g1[r]), len(g2[r]))):
            if g1[r][c] != g2[r][c]:
                return True
    return False


def _changed_cells(g1: List[List[int]], g2: List[List[int]]) -> List[Tuple[int,int]]:
    """Return (row, col) of every cell that differs between two grids."""
    changed = []
    for r in range(min(len(g1), len(g2))):
        for c in range(min(len(g1[r]), len(g2[r]))):
            if g1[r][c] != g2[r][c]:
                changed.append((r, c))
    return changed


# ============================================================================
# [L] Game-specific solvers
# ============================================================================

class _Ls20Solver:
    """[M] Navigation solver for ls20 (v13)."""

    _ARROW_ACTIONS = ['move_up', 'move_down', 'move_left', 'move_right']
    _DELTAS = {
        'move_up':    (0, -1),
        'move_down':  (0,  1),
        'move_left':  (-1, 0),
        'move_right': ( 1, 0),
    }

    def __init__(self, grid_w: int = 64, grid_h: int = 64,
                 cell_size: int = 4) -> None:
        self._gw = grid_w
        self._gh = grid_h
        self._cell = cell_size
        self._cols = grid_w // cell_size
        self._rows = grid_h // cell_size

        self._player_px: Optional[Tuple[int,int]] = None
        self._player_cell: Optional[Tuple[int,int]] = None
        self._plan: List[str] = []
        self._visited: Set[Tuple[int,int]] = set()
        self._walls: Set[Tuple[int,int]] = set()
        self._last_action: Optional[str] = None
        self._total_reward: float = 0.0
        self._steps_no_progress: int = 0
        self._hot_cells: Set[Tuple[int,int]] = set()

    def _px_to_cell(self, px: int, py: int) -> Tuple[int,int]:
        return (px // self._cell, py // self._cell)

    def _find_player(self, prev: List[List[int]],
                     curr: List[List[int]]) -> Optional[Tuple[int,int]]:
        """Detect new player position from centroid of all changes (HUD excluded)."""
        if prev is None: return self._player_px
        hud_rows = self._gh // 8
        changed = [(r, c) for r in range(hud_rows, self._gh)
                   for c in range(self._gw) if prev[r][c] != curr[r][c]]
        if not changed: return self._player_px
        avg_r = sum(r for r, c in changed) // len(changed)
        avg_c = sum(c for r, c in changed) // len(changed)
        return (avg_c, avg_r)

    def _bfs_nearest_unvisited(self, start: Tuple[int,int]) -> List[str]:
        from collections import deque
        queue = deque([(start, [])])
        seen = {start}
        while queue:
            (col, row), path = queue.popleft()
            for aname, (dc, dr) in self._DELTAS.items():
                nc, nr = col + dc, row + dr
                if not (0 <= nc < self._cols and 0 <= nr < self._rows):
                    continue
                if (nc, nr) in self._walls or (nc, nr) in seen:
                    continue
                new_path = path + [aname]
                if (nc, nr) not in self._visited:
                    return new_path
                seen.add((nc, nr))
                queue.append(((nc, nr), new_path))
        return []

    def reset_level(self) -> None:
        self._plan = []
        self._visited = set()
        self._walls = set()
        self._steps_no_progress = 0

    def next_action(self, prev_grid: Optional[List[List[int]]],
                    curr_grid: List[List[int]],
                    reward: float) -> str:
        if reward > 0:
            self._total_reward += reward
            self.reset_level()

        new_px = self._find_player(prev_grid, curr_grid)
        if new_px is not None:
            self._player_px = new_px
            new_cell = self._px_to_cell(new_px[0], new_px[1])
            if self._player_cell is not None and new_cell != self._player_cell:
                self._steps_no_progress = 0
            self._player_cell = new_cell
            self._hot_cells.add(new_cell)

        if (self._last_action is not None and prev_grid is not None
                and self._player_cell is not None):
            dc, dr = self._DELTAS.get(self._last_action, (0, 0))
            exp_col = self._player_cell[0] + dc
            exp_row = self._player_cell[1] + dr
            expected = (exp_col, exp_row)
            hud_rows = self._gh // 8
            pf_changes = sum(
                1 for r in range(hud_rows, self._gh)
                for c in range(self._gw)
                if prev_grid[r][c] != curr_grid[r][c]
            )
            if (pf_changes <= 3 and expected != self._player_cell
                    and 0 <= exp_col < self._cols
                    and 0 <= exp_row < self._rows):
                self._walls.add(expected)

        if self._player_cell is not None:
            if self._player_cell not in self._visited:
                self._steps_no_progress = 0
            self._visited.add(self._player_cell)

        self._steps_no_progress += 1
        if self._steps_no_progress > 40:
            self._steps_no_progress = 0
            self._visited.clear()
            self._plan = []

        if self._plan:
            action = self._plan.pop(0)
            self._last_action = action
            return action

        if self._player_cell is not None:
            path = self._bfs_nearest_unvisited(self._player_cell)
            if path:
                self._plan = path[1:]
                action = path[0]
                self._last_action = action
                return action

        action = random.choice(self._ARROW_ACTIONS)
        self._last_action = action
        return action


class _Ft09Solver:
    """[M] Colour-constraint solver for ft09 (v13)."""

    def __init__(self, canvas: int = 64) -> None:
        self._canvas = canvas
        self._interactive: List[Tuple[int,int]] = []
        self._probed: Set[Tuple[int,int]] = set()
        self._probe_queue: List[Tuple[int,int]] = []
        self._probe_idx: int = 0
        self._discovery_done: bool = False
        self._cell_state: Dict[Tuple[int,int], int] = {}
        self._palette_size: int = 2
        self._winning_combos: List[List[int]] = []
        self._combo_iter: Optional[object] = None
        self._queue: List[Tuple[int,int]] = []
        self._last_click: Optional[Tuple[int,int]] = None
        self._build_probe_queue()

    def _build_probe_queue(self) -> None:
        pts = []
        for y in range(2, self._canvas, 4):
            for x in range(2, self._canvas, 4):
                pts.append((x, y))
        self._probe_queue = pts

    def _detect_palette_size(self, grid: List[List[int]]) -> int:
        vals = {v for row in grid for v in row if v != 0}
        return max(2, len(vals))

    def reset_for_new_level(self) -> None:
        self._cell_state = {pos: 0 for pos in self._interactive}
        self._combo_iter = None
        self._queue = []

    def _start_combo_iter(self) -> None:
        import itertools
        n = len(self._interactive)
        if n == 0: return
        k = self._palette_size
        if k ** n > 256: self._combo_iter = None; return
        self._combo_iter = itertools.product(range(k), repeat=n)

    def _combo_to_clicks(self, combo) -> List[Tuple[int,int]]:
        clicks = []
        for pos, target in zip(self._interactive, combo):
            current = self._cell_state.get(pos, 0)
            n_clicks = (target - current) % self._palette_size
            clicks.extend([pos] * n_clicks)
        return clicks

    def next_action(self, prev_grid: Optional[List[List[int]]],
                    curr_grid: List[List[int]],
                    reward: float) -> Tuple[int, int]:
        if reward > 0:
            winning = [self._cell_state.get(pos, 0) for pos in self._interactive]
            if winning not in self._winning_combos: self._winning_combos.append(winning)
            self.reset_for_new_level()

        if prev_grid is not None and self._last_click is not None:
            changed = _changed_cells(prev_grid, curr_grid)
            if changed:
                lx, ly = self._last_click
                near = [(r, c) for r, c in changed if abs(c - lx) <= 6 and abs(r - ly) <= 6]
                if near:
                    pos = (lx, ly)
                    if pos not in self._interactive: self._interactive.append(pos); self._cell_state[pos] = 0
                    self._cell_state[pos] = (self._cell_state.get(pos, 0) + 1) % self._palette_size

        if curr_grid: self._palette_size = self._detect_palette_size(curr_grid)

        if not self._discovery_done:
            if self._probe_idx < len(self._probe_queue):
                pos = self._probe_queue[self._probe_idx]; self._probe_idx += 1
                self._last_click = pos; return pos
            self._discovery_done = True
            self._start_combo_iter()

        if self._winning_combos and not self._queue:
            target = random.choice(self._winning_combos); self._queue = self._combo_to_clicks(target)

        if not self._queue and self._combo_iter is not None:
            try:
                combo = list(next(self._combo_iter)); self._queue = self._combo_to_clicks(combo)
            except StopIteration: self._combo_iter = None; self._start_combo_iter()

        if not self._queue and self._interactive:
            self._queue = [random.choice(self._interactive)]

        if self._queue:
            pos = self._queue.pop(0); self._last_click = pos; return pos

        x, y = random.randint(0, 63), random.randint(0, 63)
        self._last_click = (x, y); return (x, y)


class _Vc33Solver:
    """[M] Constraint solver for vc33 (v13)."""

    def __init__(self, canvas: int = 64) -> None:
        self._canvas = canvas
        self._probe_queue: List[Tuple[int,int]] = []
        self._probe_idx = 0
        self._discovery_done = False
        self._responsive: List[Tuple[int,int]] = []
        self._winning_sequences: List[List[Tuple[int,int]]] = []
        self._pending_clicks = []
        self._max_clicks_per_cell = 8
        self._last_click = None
        for y in range(4, 64, 8):
            for x in range(4, 64, 8): self._probe_queue.append((x, y))
        random.shuffle(self._probe_queue)

    def next_action(self, prev_grid, curr_grid, reward) -> Tuple[int, int]:
        if reward > 0:
            if self._last_click: self._winning_sequences.append([self._last_click])
            self._pending_clicks = []
        if prev_grid is not None and self._last_click is not None:
            changed = _changed_cells(prev_grid, curr_grid)
            if changed:
                lx, ly = self._last_click
                near = [(r, c) for r, c in changed if abs(c - lx) <= 24 and abs(r - ly) <= 24]
                if 1 <= len(near) <= 256:
                    pos = (lx, ly)
                    if pos not in self._responsive: self._responsive.append(pos)
        if not self._discovery_done:
            if self._probe_idx < len(self._probe_queue):
                pos = self._probe_queue[self._probe_idx]; self._probe_idx += 1
                self._last_click = pos; return pos
            self._discovery_done = True
        if self._winning_sequences and not self._pending_clicks:
            self._pending_clicks = list(random.choice(self._winning_sequences))
        if not self._pending_clicks and self._responsive:
            chosen = random.choice(self._responsive)
            self._pending_clicks = [chosen] * random.randint(1, self._max_clicks_per_cell)
        if self._pending_clicks:
            pos = self._pending_clicks.pop(0); self._last_click = pos; return pos
        x, y = random.randint(0, 63), random.randint(0, 63)
        self._last_click = (x, y); return (x, y)


class _FrameChangeDetector:
    def __init__(self, canvas: int = 64) -> None:
        self._canvas = canvas
        self._hit_map: List[List[float]] = [[0.0] * canvas for _ in range(canvas)]
        self._total_hits = 0.0
        self._no_change_streak = 0
        self._dir_hits: Dict[str, float] = {
            'move_up': 0.0, 'move_down': 0.0,
            'move_left': 0.0, 'move_right': 0.0,
        }
    def record_click(self, changed: bool, xy: Optional[Tuple[int, int]]) -> None:
        if changed and xy is not None:
            cx, cy = xy
            for dy in range(-4, 5):
                for dx in range(-4, 5):
                    nx, ny = cx + dx, cy + dy
                    if 0 <= nx < self._canvas and 0 <= ny < self._canvas:
                        w = math.exp(-(dx*dx + dy*dy) / 8.0)
                        self._hit_map[ny][nx] += w; self._total_hits += w
            self._no_change_streak = 0
        else: self._no_change_streak += 1
    def record_arrow(self, changed: bool, direction: str, player_px: Optional[Tuple[int,int]]=None) -> None:
        if changed and direction in self._dir_hits:
            self._dir_hits[direction] += 1.0; self._no_change_streak = 0
            if player_px:
                px, py = player_px
                if 0 <= px < self._canvas and 0 <= py < self._canvas:
                    self._hit_map[py][px] += 1.0; self._total_hits += 1.0
        else: self._no_change_streak += 1
    def change_biased_click(self) -> ARC3Action:
        if self._total_hits < 1.0 or self._no_change_streak >= 15:
            return ARC3Action(action_type='place', x=random.randint(0, 63), y=random.randint(0, 63))
        flat = [v for row in self._hit_map for v in row]
        total = sum(flat); r = random.random() * total; cumulative = 0.0
        for idx, v in enumerate(flat):
            cumulative += v
            if cumulative >= r: y, x = divmod(idx, self._canvas); return ARC3Action(action_type='place', x=x, y=y)
        return ARC3Action(action_type='place', x=random.randint(0, 63), y=random.randint(0, 63))
    def best_arrow(self) -> str:
        total = sum(self._dir_hits.values())
        if total < 1.0: return random.choice(list(self._dir_hits.keys()))
        weights = list(self._dir_hits.values()); keys = list(self._dir_hits.keys())
        r = random.random() * total; cumulative = 0.0
        for k, w in zip(keys, weights):
            cumulative += w
            if cumulative >= r: return k
        return keys[-1]
    def decay(self, factor: float = 0.9) -> None:
        for r in range(self._canvas):
            for c in range(self._canvas): self._hit_map[r][c] *= factor
        self._total_hits *= factor; for k in self._dir_hits: self._dir_hits[k] *= factor
    @property
    def hot_cell_count(self) -> int:
        return sum(1 for row in self._hit_map for v in row if v > 0.01)


class _GridScanner:
    def __init__(self, canvas: int = 64, divisions: int = 16) -> None:
        block = canvas // divisions
        self._centres = [(block*c + block//2, block*r + block//2) for r in range(divisions) for c in range(divisions)]
        random.shuffle(self._centres); self._idx = 0
    def next_click(self) -> ARC3Action:
        x, y = self._centres[self._idx % len(self._centres)]; self._idx += 1
        return ARC3Action(action_type='place', x=x, y=y)
    def coverage_fraction(self) -> float:
        return min(self._idx, len(self._centres)) / len(self._centres)


class PrometheusARC3LiveEnv:
    def __init__(self, env_wrapper, game_id: str) -> None:
        self._env = env_wrapper; self.game_type = game_id
        self._game_allowed = _GAME_ALLOWED_ACTIONS.get(game_id)
        self.change_detector = _FrameChangeDetector()
        self._ls20_solver = _Ls20Solver() if game_id == 'ls20' else None
        self._ft09_solver = _Ft09Solver() if game_id == 'ft09' else None
        self._vc33_solver = _Vc33Solver() if game_id == 'vc33' else None
        self.total_actions = 0; self.total_levels = 0; self._window_actions = 0
        self._window_levels_start = 0; self._available = list(range(1, 8))
        self._last_grid = None; self._prev_grid = None; self._last_raw_frame = None
        self._start_session()
    def _start_session(self) -> None:
        frame = self._env.reset(); self._last_raw_frame = frame
        self._last_grid = _frame_to_grid(frame); self._prev_grid = None
        self.total_levels = int(getattr(frame, 'levels_completed', 0) or 0)
    def begin_window(self) -> ARC3Observation:
        self._window_actions = 0; self._window_levels_start = self.total_levels
        self.change_detector.decay(0.85); return self._raw_to_obs(self._last_raw_frame, 0)
    def _raw_to_obs(self, frame, step) -> ARC3Observation:
        grid = _frame_to_grid(frame); lvl = int(getattr(frame, 'levels_completed', self.total_levels) or self.total_levels)
        state = getattr(frame, 'state', None); done = False
        if TOOLKIT_AVAILABLE and state is not None:
            try: done = state in (GameState.WIN, GameState.GAME_OVER)
            except: done = str(state).upper() in ('WIN', 'GAME_OVER')
        return ARC3Observation.from_grid_list(grid, score=float(lvl), done=done, step=step)
    def solver_action(self, reward) -> Optional[ARC3Action]:
        if self._ls20_solver: return ARC3Action(action_type=self._ls20_solver.next_action(self._prev_grid, self._last_grid, reward))
        if self._ft09_solver: x, y = self._ft09_solver.next_action(self._prev_grid, self._last_grid, reward); return ARC3Action(action_type='place', x=x, y=y)
        if self._vc33_solver: x, y = self._vc33_solver.next_action(self._prev_grid, self._last_grid, reward); return ARC3Action(action_type='place', x=x, y=y)
        return None
    def step(self, action, override_reward=0.0) -> Tuple[ARC3Observation, float]:
        prev_grid = self._last_grid; prev_levels = self.total_levels
        ga_name = _TO_GA.get(action.action_type, 'ACTION1'); ga_num = int(ga_name.replace('ACTION', ''))
        action_data = {'x': int(action.x), 'y': int(action.y)} if ga_num == 6 else None
        self._window_actions += 1; self.total_actions += 1
        try:
            frame = self._env.step(getattr(GameAction, ga_name), data=action_data); self._last_raw_frame = frame
        except: frame = self._last_raw_frame
        new_grid = _frame_to_grid(frame); changed = _grids_differ(prev_grid, new_grid)
        if ga_num == 6: self.change_detector.record_click(changed, (action.x, action.y))
        elif action.action_type in self.change_detector._dir_hits:
            px = self._ls20_solver._player_px if self._ls20_solver else None
            self.change_detector.record_arrow(changed, action.action_type, px)
        self._prev_grid = prev_grid; self._last_grid = new_grid
        self.total_levels = int(getattr(frame, 'levels_completed', prev_levels) or prev_levels)
        reward = float(self.total_levels - prev_levels)
        return self._raw_to_obs(frame, self._window_actions), reward
    @property
    def available_action_types(self) -> List[str]: return self._game_allowed or ['move_up']
    @property
    def window_levels(self) -> int: return self.total_levels - self._window_levels_start

class PatchedExplorationPolicy(ARC3ExplorationPolicy):
    _STRATEGIES = ARC3ExplorationPolicy._STRATEGIES + ['grid_scan', 'frame_change_guided', 'repeat_best']
    def __init__(self, mutation_rate=0.05, scanner=None, change_detector=None, model_guided_temp=0.3):
        super().__init__(mutation_rate=mutation_rate); self._scanner = scanner; self._change_detector = change_detector
        self._model_guided_temp = model_guided_temp; n = len(self._STRATEGIES); self._probs = {s: 1.0/n for s in self._STRATEGIES}
    def select_action(self, obs, world_model, goal_inferrer, solved_episodes=None, available_types=None, strategy=None):
        if strategy is None: strategy = self.active_strategy
        if strategy == 'grid_scan': return self._scanner.next_click() if self._scanner else ARC3Action(action_type='place', x=random.randint(0,63), y=random.randint(0,63))
        if strategy == 'frame_change_guided': return self._change_detector.change_biased_click() if self._change_detector else ARC3Action(action_type='place', x=random.randint(0,63), y=random.randint(0,63))
        return super().select_action(obs, world_model, goal_inferrer, solved_episodes)

class PatchedStrangeLoopAgent(ARC3StrangeLoopAgent):
    def __init__(self, window_steps=100, mutation_rate=0.05, fitness_threshold=0.5, scanner=None, change_detector=None):
        super().__init__(max_steps_per_episode=window_steps, mutation_rate=mutation_rate, fitness_threshold=fitness_threshold)
        self.policy = PatchedExplorationPolicy(mutation_rate=mutation_rate, scanner=scanner, change_detector=change_detector)
        self.episode_logs = []
    def run_episode(self, env) -> ARC3Episode:
        self._episode_count += 1; obs = env.begin_window(); episode = ARC3Episode(game_id=env.game_type, level=self._episode_count)
        strategy = self.policy.active_strategy; prev_obs = None; total_reward = 0.0; step_log = []
        for step_idx in range(self.max_steps_per_episode):
            solver_act = env.solver_action(0.0 if step_idx==0 else step_log[-1]['reward'])
            if solver_act: action = solver_act; used_solver = True
            else: action = self.policy.select_action(obs, self.world_model, self.goal_inferrer, strategy=strategy); used_solver = False
            next_obs, reward = env.step(action); total_reward += reward
            self.world_model.update(obs, action, next_obs, reward); self.goal_inferrer.observe(next_obs, obs, reward)
            episode.record(obs, action, reward); step_log.append({'strategy': 'solver' if used_solver else strategy, 'reward': reward})
            prev_obs = obs; obs = next_obs
        episode.solved = (total_reward >= self.fitness_threshold); self.policy.record_episode(strategy, total_reward); self.policy.mutate()
        self.episode_logs.append({'episode': self._episode_count, 'strategy': strategy, 'total_reward': total_reward, 'solved': episode.solved, 'hot_cells': env.change_detector.hot_cell_count})
        return episode

def run_live_game(game_id='ls20', n_windows=30, window_steps=100, mutation_rate=0.10, fitness_threshold=0.5, verbose=True):
    if not TOOLKIT_AVAILABLE: return None
    print(f'Connecting to ARC-AGI-3 API [{game_id}]...')
    arc = arc_agi.Arcade(); env_w = arc.make(game_id, render_mode=None)
    live_env = PrometheusARC3LiveEnv(env_w, game_id); scanner = _GridScanner(divisions=16)
    agent = PatchedStrangeLoopAgent(window_steps=window_steps, mutation_rate=mutation_rate, fitness_threshold=fitness_threshold, scanner=scanner, change_detector=live_env.change_detector)
    episodes = []
    for win in range(n_windows):
        t0 = time.time(); ep = agent.run_episode(live_env); episodes.append(ep)
        if verbose:
            log = agent.episode_logs[-1]
            print(f'  Win {win+1:>2}/{n_windows}: levels=+{ep.total_score:.0f}(total={live_env.total_levels})  score={ep.total_score:.1f}  steps={ep.steps}  solved={ep.solved}  strategy={log["strategy"]}  hot={log["hot_cells"]}  ({time.time()-t0:.1f}s)')
    sr = sum(1 for e in episodes if e.solved) / len(episodes)
    return {'game_id': game_id, 'solve_rate': sr, 'mean_score': sum(e.total_score for e in episodes)/len(episodes), 'source': 'live_api', 'world_model': agent.world_model.to_dict(), 'goal_inferrer': agent.goal_inferrer.to_dict(), 'policy': agent.policy.to_dict(), 'entanglement_index': agent.entanglement_index}

print('Bridge v13 ready.')
print(f'Toolkit available: {TOOLKIT_AVAILABLE}')
print(f'Game-specific action filters: {_GAME_ALLOWED_ACTIONS}')
print(f'Solvers: ls20=_Ls20Solver  ft09=_Ft09Solver  vc33=_Vc33Solver')


In [ ]:
# ── Run Prometheus on ARC-AGI-3 ────────────
#
# Bridge v14: Dynamic game loading + CNN pattern recognition.

SELECTED_GAMES = ["ls20", "ft09", "vc33"]
if ARC_API_KEY and TOOLKIT_AVAILABLE:
    try:
        arc = arc_agi.Arcade()
        all_envs = arc.get_environments()
        SELECTED_GAMES = [e.game_id for e in all_envs]
        print(f"API Key detected: Loading all {len(SELECTED_GAMES)} games.")
    except:
        print("API Key failed: Falling back to public games.")

N_WINDOWS     = 60
WINDOW_STEPS  = 200

live_results = {}
t_start = time.time()

# Run on first 5 games to keep demo snappy if all loaded
GAMES_TO_RUN = SELECTED_GAMES[:5] if len(SELECTED_GAMES) > 3 else SELECTED_GAMES

for game_id in GAMES_TO_RUN:
    print(f"
{"="*55}")
    print(f"  Game: {game_id}")
    print("="*55)
    result = run_live_game(game_id=game_id, n_windows=N_WINDOWS, window_steps=WINDOW_STEPS, mutation_rate=0.10, fitness_threshold=0.5, verbose=True)
    if result:
        live_results[game_id] = result
        print(f"  --> solve rate: {result["solve_rate"]:.0%}  mean score: {result["mean_score"]:.3f}")

print(f"
Total time: {time.time()-t_start:.1f}s")


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import math

def visualize_live_results(live_results: dict):
    if not live_results: return
    games = list(live_results.keys())
    colours = ['#3498db', '#e74c3c', '#2ecc71']
    fig, ax = plt.subplots(1, 1, figsize=(10, 6))
    sr = [live_results[g]['solve_rate'] * 100 for g in games]
    bars = ax.bar(games, sr, color=colours[:len(games)], edgecolor='black')
    ax.set_ylabel('Solve rate (%)')
    ax.set_title('Live ARC-AGI-3 Solve Rates (Bridge v13)')
    plt.show()

visualize_live_results(live_results)


In [ ]:
# ── (Optional) List all available ARC-AGI-3 games ───────
if TOOLKIT_AVAILABLE:
    try:
        arc = arc_agi.Arcade()
        games = arc.get_environments()
        print(f'Available games ({len(games)} total):')
        for g in games: print(f'  {g.game_id:8s}  {g.title[:45]:45s}')
    except: pass
